# Base 4 — Limpeza e preparação dos dados

**Base:** Jena Climate, série meteorológica em frequência nominal de 10 minutos.  
**Alvo definido na documentação:** `T (degC)`.

O notebook trata timestamps duplicados, códigos `-9999`, ordenação e irregularidades temporais. O alvo não é imputado com informação futura.


In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

DATA_PATH = Path("../../bases/grupo4/grupo4.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {DATA_PATH.resolve()}\n"
        "Coloque este notebook em trabalho/notebooks/exploracao/"
    )

df_raw = pd.read_csv(DATA_PATH)
print("Arquivo:", DATA_PATH)
print("SHA-256:", hashlib.sha256(DATA_PATH.read_bytes()).hexdigest())
print("Dimensão:", df_raw.shape)
display(df_raw.head())


Arquivo: ..\..\bases\grupo4\grupo4.csv
SHA-256: 14539fa8e185a53e97f13936716472243570e310dfb8569495fd0f7bb24f56cf
Dimensão: (420551, 15)


,Date Time,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
0,01.01.2009 00:10:00,996.52,-8.02,265.40,-8.90,93.3,3.33,3.11,0.22,1.94,3.12,1307.75,1.03,1.75,152.3
1,01.01.2009 00:20:00,996.57,-8.41,265.01,-9.28,93.4,3.23,3.02,0.21,1.89,3.03,1309.80,0.72,1.50,136.1
2,01.01.2009 00:30:00,996.53,-8.51,264.91,-9.31,93.9,3.21,3.01,0.20,1.88,3.02,1310.24,0.19,0.63,171.6
3,01.01.2009 00:40:00,996.51,-8.31,265.12,-9.07,94.2,3.26,3.07,0.19,1.92,3.08,1309.19,0.34,0.50,198.0
4,01.01.2009 00:50:00,996.51,-8.27,265.15,-9.04,94.1,3.27,3.08,0.19,1.92,3.09,1309.00,0.32,0.63,214.3


In [2]:
# 1. Conversão da data e dos códigos de ausência
df = df_raw.copy()

df["Date Time"] = pd.to_datetime(
    df["Date Time"],
    format="%d.%m.%Y %H:%M:%S",
    errors="coerce"
)

numeric_cols = [c for c in df.columns if c != "Date Time"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# -9999 é código de ausência/invalidez, não uma medição real.
invalid_counts = (df[numeric_cols] == -9999).sum()
df[numeric_cols] = df[numeric_cols].replace(-9999, np.nan)

df = df.sort_values("Date Time", kind="stable").reset_index(drop=True)

quality = pd.DataFrame({
    "métrica": [
        "linhas", "colunas", "datas nulas",
        "timestamps duplicados", "códigos -9999 encontrados"
    ],
    "valor": [
        len(df), df.shape[1], df["Date Time"].isna().sum(),
        df["Date Time"].duplicated().sum(), int(invalid_counts.sum())
    ]
})
display(quality)
display(invalid_counts[invalid_counts > 0].rename("quantidade_-9999").to_frame())


,métrica,valor
0,linhas,420551
1,colunas,15
2,datas nulas,0
3,timestamps duplicados,327
4,códigos -9999 encontrados,38


,quantidade_-9999
wv (m/s),18
max. wv (m/s),20


In [3]:
# 2. Consolidação dos timestamps duplicados
# Para cada timestamp repetido, agregamos as medições numéricas pela média.
# Isso evita manter duas observações para o mesmo instante.
before = len(df)

df_clean = (
    df.groupby("Date Time", as_index=False)[numeric_cols]
      .mean()
      .sort_values("Date Time")
      .reset_index(drop=True)
)

print("Linhas antes:", before)
print("Linhas após consolidar timestamps:", len(df_clean))
print("Duplicados restantes:", df_clean["Date Time"].duplicated().sum())


Linhas antes: 420551
Linhas após consolidar timestamps: 420224
Duplicados restantes: 0


In [4]:
# 3. Diagnóstico da grade temporal de 10 minutos
idx = pd.DatetimeIndex(df_clean["Date Time"])
diffs = idx.to_series().diff().dropna()

expected_step = pd.Timedelta(minutes=10)
irregular = diffs[diffs != expected_step]

print("Intervalos totais:", len(diffs))
print("Intervalos de 10 min:", int((diffs == expected_step).sum()))
print("Intervalos irregulares:", len(irregular))
display(irregular.sort_values(ascending=False).head(20))

# Reindexação apenas para tornar as lacunas explícitas.
regular_index = pd.date_range(
    start=idx.min(),
    end=idx.max(),
    freq="10min"
)
df_regular = df_clean.set_index("Date Time").reindex(regular_index)
df_regular.index.name = "Date Time"

print("Linhas na grade regular:", len(df_regular))
print("Timestamps criados pela reindexação:", len(df_regular) - len(df_clean))


Intervalos totais: 420223
Intervalos de 10 min: 420218
Intervalos irregulares: 5


Date Time
2016-10-28 12:50:00   3 days 02:20:00
2014-09-25 09:00:00   0 days 16:00:00
2009-10-08 10:10:00   0 days 00:30:00
2013-05-16 09:10:00   0 days 00:20:00
2014-07-30 08:20:00   0 days 00:20:00
Name: Date Time, dtype: timedelta64[ns]

Linhas na grade regular: 420768
Timestamps criados pela reindexação: 544


In [5]:
# 4. Calendário e preparação para EDA/modelagem
prepared = df_regular.reset_index()

prepared["hour"] = prepared["Date Time"].dt.hour
prepared["day_of_week"] = prepared["Date Time"].dt.dayofweek
prepared["month"] = prepared["Date Time"].dt.month

prepared["hour_sin"] = np.sin(
    2 * np.pi * (prepared["Date Time"].dt.hour * 60 + prepared["Date Time"].dt.minute) / (24 * 60)
)
prepared["hour_cos"] = np.cos(
    2 * np.pi * (prepared["Date Time"].dt.hour * 60 + prepared["Date Time"].dt.minute) / (24 * 60)
)
prepared["month_sin"] = np.sin(2 * np.pi * prepared["month"] / 12)
prepared["month_cos"] = np.cos(2 * np.pi * prepared["month"] / 12)

# Direção do vento é circular.
wind_rad = np.deg2rad(prepared["wd (deg)"])
prepared["wd_sin"] = np.sin(wind_rad)
prepared["wd_cos"] = np.cos(wind_rad)

display(prepared.head())
print("Ausências após reindexação:")
display(prepared.isna().sum().sort_values(ascending=False).head(20))


,Date Time,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),...,wd (deg),hour,day_of_week,month,hour_sin,hour_cos,month_sin,month_cos,wd_sin,wd_cos
0,2009-01-01 00:10:00,996.52,-8.02,265.40,-8.90,93.3,3.33,3.11,0.22,1.94,...,152.3,0,3,1,0.043619,0.999048,0.5,0.866025,0.464842,-0.885394
1,2009-01-01 00:20:00,996.57,-8.41,265.01,-9.28,93.4,3.23,3.02,0.21,1.89,...,136.1,0,3,1,0.087156,0.996195,0.5,0.866025,0.693402,-0.720551
2,2009-01-01 00:30:00,996.53,-8.51,264.91,-9.31,93.9,3.21,3.01,0.20,1.88,...,171.6,0,3,1,0.130526,0.991445,0.5,0.866025,0.146083,-0.989272
3,2009-01-01 00:40:00,996.51,-8.31,265.12,-9.07,94.2,3.26,3.07,0.19,1.92,...,198.0,0,3,1,0.173648,0.984808,0.5,0.866025,-0.309017,-0.951057
4,2009-01-01 00:50:00,996.51,-8.27,265.15,-9.04,94.1,3.27,3.08,0.19,1.92,...,214.3,0,3,1,0.216440,0.976296,0.5,0.866025,-0.563526,-0.826098


Ausências após reindexação:


max. wv (m/s)      564
wv (m/s)           562
sh (g/kg)          544
wd_sin             544
wd (deg)           544
p (mbar)           544
rho (g/m**3)       544
H2OC (mmol/mol)    544
wd_cos             544
VPdef (mbar)       544
VPact (mbar)       544
VPmax (mbar)       544
rh (%)             544
Tdew (degC)        544
Tpot (K)           544
T (degC)           544
hour                 0
day_of_week          0
month                0
hour_sin             0
dtype: int64

In [6]:
# 5. Separação cronológica para futura modelagem
target_df = prepared.dropna(subset=["T (degC)"]).copy()

split = int(len(target_df) * 0.80)
train_df = target_df.iloc[:split].copy()
test_df = target_df.iloc[split:].copy()

split_info = pd.DataFrame({
    "conjunto": ["treino", "teste"],
    "linhas": [len(train_df), len(test_df)],
    "proporção": [len(train_df)/len(target_df), len(test_df)/len(target_df)],
    "início": [train_df["Date Time"].min(), test_df["Date Time"].min()],
    "fim": [train_df["Date Time"].max(), test_df["Date Time"].max()]
})
display(split_info)
assert train_df["Date Time"].max() < test_df["Date Time"].min()

prepared.to_csv("base4_limpa_preparada.csv", index=False)
train_df.to_csv("base4_treino_preparada.csv", index=False)
test_df.to_csv("base4_teste_preparada.csv", index=False)

print("Arquivos CSV salvos na pasta de execução do notebook.")


,conjunto,linhas,proporção,início,fim
0,treino,336179,0.8,2009-01-01 00:10:00,2015-05-25 06:20:00
1,teste,84045,0.2,2015-05-25 06:30:00,2017-01-01 00:00:00


Arquivos CSV salvos na pasta de execução do notebook.


## Decisões de preparação

- `Date Time` foi convertido para `datetime` usando `%d.%m.%Y %H:%M:%S`.
- Os códigos `-9999` foram convertidos para `NaN`.
- Timestamps duplicados foram consolidados pela média das variáveis numéricas.
- A série foi reindexada para uma grade explícita de 10 minutos; as lacunas ficaram como `NaN`.
- O alvo `T (degC)` não foi interpolado. Isso evita introduzir informação futura antes da modelagem.
- `wd (deg)` foi preparado como variável circular via seno/cosseno.
- A separação treino/teste é cronológica.
- Para Holt-Winters, somente `T (degC)` deve entrar no modelo.
